### CreateSources (post-cutover, oxjob #548)

The sources registry lives in the **openalex-sources** Heroku Postgres and is maintained by
that app's feed / merge / curation jobs. This notebook no longer *builds* sources — it
materializes run-consistent Delta snapshots of the federated Source registry and canonical
endpoint relationships. Downstream consumers retain their stable table interfaces. Runs as
a plain SQL-warehouse task (`source: GIT`) — the DLT pipeline is retired.

Source snapshot contract: `webpage` = `homepage_url`; JSONB columns parsed to typed arrays;
date columns cast to string. It intentionally does **not** carry the legacy
`sources.endpoint_id` relationship. `issns` = registry array **verbatim** — NULL when the
source has no ISSNs (uniform convention across the registry and this snapshot; the mixed
NULL/[] compat shim via `legacy_empty_issns_sources` was retired 2026-07-09, verified
churn-free 2026-07-10 — the works content hash in CreateWorksEnriched is []-blind).
`issn_l` carries the registry name directly; the inverted legacy alias `issn` was dropped
2026-07-10 (C9) after CreateWorksBase + CreateSourcesApi were repointed to `issn_l`.
Post-D1 (merge-column removal, oxjob #629 endgame) the registry holds ACTIVE sources only:
merge losers are deleted and recorded in `public.source_merge` (audit ledger, kept). The
Source snapshot LEFT ANTI JOINs `source_merge` losers as belt-and-braces so a loser can
never resurface in the mirror; the legacy `merge_into_id`/`merge_into_date` columns are
dropped from the contract and consumers read the mirror unfiltered. NOTE: any future
unmerge-by-restore must also delete the loser's `source_merge` row(s), or the anti-join
will hide the restored source.

`openalex.sources.endpoint_to_source` is canonical-first: `endpoint.source_id` wins whenever
present. Five exact-ID `COALESCE` fallbacks preserve legacy cutover behavior while their
canonical values remain NULL. They are transitional compatibility rows, not relationship
authority; this notebook never reads or writes `source_endpoint`.

**2026-09-01 (oxjob 83.13):** registry table is `oai_pmh_endpoint`; `source_id` NOT NULL; fallbacks retired.


In [ ]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.sources.sources')
-- 60-day time travel; must live in this DDL (not ALTER) because the daily
-- CREATE OR REPLACE resets any properties it doesn't restate
TBLPROPERTIES (
  'delta.logRetentionDuration' = 'interval 60 days',
  'delta.deletedFileRetentionDuration' = 'interval 60 days'
)
AS SELECT
  s.id,
  display_name,
  issn_l,
  publisher,
  homepage_url AS webpage,
  is_oa,
  type,
  from_json(apc_prices, 'array<struct<price:int,currency:string>>') AS apc_prices,
  from_json(apc_usd_by_year, 'map<string,int>') AS apc_usd_by_year,
  from_json(apc_prices_by_year, 'map<string,array<struct<price:decimal(18,4),currency:string>>>') AS apc_prices_by_year,
  is_society_journal,
  from_json(societies, 'array<struct<url:string,organization:string>>') AS societies,
  apc_usd,
  fatcat_id,
  wikidata_id,
  crossref_id,
  country,
  country_code,
  from_json(alternate_titles, 'array<string>') AS alternate_titles,
  publisher_id,
  institution_id,
  is_core,
  CAST(updated_date AS string) AS updated_date,
  CAST(created_date AS string) AS created_date,
  display_name_before_override,
  override_timestamp,
  datacite_id,
  -- verbatim registry value: NULL when the source has no ISSNs (the registry
  -- derives issns via array_agg, which never yields []). The works hash is
  -- []-blind (CreateWorksEnriched), so the legacy-[] cohort converges to NULL
  -- without churn (oxjob #548; compat table legacy_empty_issns_sources retired)
  s.issns,
  is_in_doaj,
  is_in_doaj_start_year,
  doaj_license,
  is_in_scielo,
  is_ojs,
  is_oa_high_oa_rate,
  high_oa_rate_start_year,
  is_fully_open_in_jstage,
  sample_pmh_record,
  COALESCE(from_json(datacite_ids, 'array<string>'), array()) AS datacite_ids,
  is_preprint_repository
FROM openalex_sources.public.sources s
-- D1: exclude merge losers without referencing the (dropped) merge columns.
-- Pre-migration this filters the tombstoned rows (loser set == tombstone set);
-- post-migration the losers are deleted and this is a no-op safety net.
LEFT ANTI JOIN openalex_sources.public.source_merge m ON s.id = m.loser_id

In [ ]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.sources.endpoint_to_source')
-- Derived cache of the registry binding. endpoint.source_id is NOT NULL and FK-enforced
-- since openalex-sources migration 035 (2026-09-01, oxjob 83.13); the five legacy-only
-- COALESCE fallbacks that stood in for unbound rows during cutover are retired (their
-- endpoints are now bound or deleted). Table renamed endpoint -> oai_pmh_endpoint (mig 036).
-- 60-day time travel; restated here because CREATE OR REPLACE resets properties.
TBLPROPERTIES (
  'delta.logRetentionDuration' = 'interval 60 days',
  'delta.deletedFileRetentionDuration' = 'interval 60 days'
)
AS
WITH resolved AS (
  SELECT
    e.id AS endpoint_id,
    e.source_id
  FROM openalex_sources.public.oai_pmh_endpoint e
)
SELECT
  endpoint_id,
  source_id
FROM resolved
WHERE source_id IS NOT NULL

In [ ]:
SELECT
  COUNT(*) AS total_active,  -- mirror is active-only post-D1 (losers deleted/anti-joined)
  COUNT_IF(is_in_doaj) AS in_doaj,
  MAX(id) AS max_id
FROM identifier('openalex' || :env_suffix || '.sources.sources')